# Suffolk County ZIP Code Desirability — Exploratory Data Analysis
**Data:** 2023 U.S. Census Bureau (ACS 5-Year) + NeighborhoodScout Crime Data  
**Scope:** 33 ZIP codes across Suffolk County, MA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load Data

In [ ]:
df = pd.read_excel('../data/suffolk_zipcode_data.xlsx')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Missing Values & Data Quality

In [ ]:
print(df.isnull().sum())

## 3. Distribution of Raw Variables

In [ ]:
raw_cols = ['Population', 'Median Income', 'Education Level (%)', 'Crime Rate (%)', 'Median Housing Cost']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(raw_cols):
    axes[i].hist(df[col], bins=10, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
fig.suptitle('Distribution of Raw Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Composite Score Distribution

In [ ]:
composite_col = [c for c in df.columns if 'composite' in c.lower() or 'desirability' in c.lower() or 'score' in c.lower() and 'income' not in c.lower()]
print('Score columns found:', composite_col)

In [ ]:
score_col = composite_col[0] if composite_col else 'COMPOSITE SCORE'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[score_col], bins=10, color='teal', edgecolor='white')
axes[0].set_title('Composite Score Distribution')
axes[0].set_xlabel('Score')

top10 = df.nlargest(10, score_col)[['ZIP Code', 'City', score_col]]
axes[1].barh(top10['ZIP Code'].astype(str) + ' - ' + top10['City'], top10[score_col], color='teal')
axes[1].invert_yaxis()
axes[1].set_title('Top 10 ZIP Codes by Composite Score')
axes[1].set_xlabel('Score')

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['ZIP Code', 'LATITUDE', 'LONGITUDE'], errors='ignore')

plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Matrix — All Numeric Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Score Component Breakdown

In [ ]:
score_cols = [c for c in df.columns if 'SCORE' in c.upper()]
print('Score columns:', score_cols)

In [ ]:
component_cols = [c for c in score_cols if 'COMPOSITE' not in c.upper()]

fig, axes = plt.subplots(1, len(component_cols), figsize=(15, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

for i, col in enumerate(component_cols):
    axes[i].scatter(df[col], df[score_col], color=colors[i % len(colors)], alpha=0.7)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Composite Score')
    axes[i].set_title(col.replace(' SCORE', ''))

fig.suptitle('Score Components vs. Composite Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Income vs. Crime Rate by ZIP

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    df['Median Income'], df['Crime Rate (%)'],
    c=df[score_col], cmap='RdYlGn', s=100, edgecolors='grey', linewidth=0.5
)
plt.colorbar(scatter, label='Composite Score')
plt.xlabel('Median Household Income ($)')
plt.ylabel('Crime Rate (%)')
plt.title('Income vs. Crime Rate (colored by Composite Score)', fontweight='bold')

for _, row in df.iterrows():
    plt.annotate(str(row['ZIP Code']), (row['Median Income'], row['Crime Rate (%)']),
                 fontsize=7, ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 8. Full ZIP Code Rankings

In [ ]:
ranking = df[['ZIP Code', 'City', 'Median Income', 'Education Level (%)', 'Crime Rate (%)', 'Median Housing Cost', score_col]]\
    .sort_values(score_col, ascending=False)\
    .reset_index(drop=True)
ranking.index += 1
ranking

## 9. Key Takeaways

- **Top ZIP codes** cluster in downtown Boston (02108, 02210, 02116) driven by high income and education scores
- **Crime rate** shows the strongest inverse relationship with composite score
- **Housing cost** correlates positively with income but negatively with crime — high-cost areas tend to be safer
- **Population density** has mixed effects — dense ZIPs score variably depending on income/crime profile
- Composite scoring formula: `(Income×25%) + (Education×20%) + (Housing×20%) + (Crime×20%) + (Population×15%)`